# Extra: selection of TF-IDF components based on entropy

This notebook is a small extension of the original project. Here we want to further analyze the network built using tf-idf embeddings of abstracts. In particular, since tf-idf embedding vectors have a huge number of components, we want to see if filtering some of these components can lead to a reduction of the noise present in the embedding vectors, and thus bring to an improvement of classification performances.

The selection of components is based on entropy: for each component of the vector embeddings, the information entropy is calculated across all papers is calculated using:

\begin{equation}
    H = -\sum_{i=1}^N p_ilog(p_i)
\end{equation}

where $N = 25877$ is the total number of papers, while $p_i$ represents the value of a certain component of embedding vectors in paper $i$ when the values of that component are normalized across all papers so that $\sum_{i=1}^N p_i = 1$.

After calculating the value of entropy for all embedding components, only the components with entropy higher than a certain threshold are kept. Finally, the distance matrix, the adjacency matrix and the corresponding network are built from tf-idf embeddings filtered in this way, and all the algorithms already employed in the original project are tried to divide the network into communities, to see if we have an improvement of classification after the selection of components.

## Entropy calculation

First of all we want to compute the entropy of all components of embedding vectors, and save in a npz file the array containing all values of entropy.

In [8]:
from components_selection import entropy_component
from scipy.sparse import load_npz, save_npz
import numpy as np
import pandas as pd
import networkx as nx
import subprocess
import pickle
from collections import Counter

In [2]:
tf_idf_embeddings = load_npz("../embeddings/abstract_embeddings_tfidf.npz")

In [ ]:
for i in range(223):
    #select 250 columns, since 223 * 250 = 55750, number of columns of the original matrix
    batch_of_columns = tf_idf_embeddings[:, (i * 250):((i+1) * 250)].toarray()

    entropy = entropy_component(batch_of_columns)

    #append the arrays obtained in the different cycles
    if i == 0:
        full_entropy_array = entropy
    else:
        full_entropy_array = np.concatenate((full_entropy_array, entropy))

#save the array in the npz format
np.savez("./entropy_array.npz", full_entropy_array)

/home/riccardo/uni_projects/complex_networks/Abstract_network/extra/components_selection.py:32: RuntimeWarning: divide by zero encountered in log
  embedding_matrix_normalized = np.where(np.isclose(embedding_matrix_normalized, 0.), 0., (embedding_matrix_normalized * np.log(embedding_matrix_normalized)) * (-1))
/home/riccardo/uni_projects/complex_networks/Abstract_network/extra/components_selection.py:32: RuntimeWarning: invalid value encountered in multiply
  embedding_matrix_normalized = np.where(np.isclose(embedding_matrix_normalized, 0.), 0., (embedding_matrix_normalized * np.log(embedding_matrix_normalized)) * (-1))


Now load the array with entropies, and filter the components of the tf-idf embeddings keeping only those with entropy greater than 1 (this threshold on entropy is selected arbitrarily, later we will see how different values of threshold influence the structure of the resulting network).

In [3]:
entropy_array = np.load("./entropy_array.npz")['arr_0']

In [9]:
print(np.max(entropy_array))
print(np.min(entropy_array))

10.028510818552421
0.0


As a first analysis, we can print the maximum and minimum values of entropy of the embedding components: as we expect, the minimum value of entropy is 0: this happens when the component is different from 0 only in one paper (so it is a word that appears only in a certain paper).

## Build network

Now we can build the distance matrix, the adjacency matrix, the relative network using the same functions we used for the network with all the components.

Using always the threshold distance of 0.2, we try different values of threshold entropy to filter the components of the embeddings. In particular, we use ten different thresholds in the range 1-8 (the range is selected considering the minimum and maximum value of entropy).

In [1]:
from components_selection import sweep_entropy_threshold

sweep_entropy_threshold()

Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration


In [4]:
pd.read_csv("./results/connected_components_entropy.csv")

,Unnamed: 0,Threshold,Embedding_components,Connected_components,Largest_component
0,0,0.5,25421.0,25686.0,3.0
1,1,1.0,32761.0,20570.0,18.0
2,2,1.5,38262.0,13030.0,7952.0
3,3,2.0,42100.0,7602.0,17378.0
4,4,2.5,45040.0,4603.0,20970.0
5,5,3.0,47281.0,2845.0,22881.0
6,6,3.5,49138.0,1975.0,23807.0
7,7,4.0,50686.0,1713.0,24072.0
8,8,4.5,51960.0,1823.0,23944.0
9,9,5.0,53013.0,2053.0,23708.0


We use a threshold of 2.5 (so that we eliminate almost 10000 components, so almost 20% of the total components),and we try different values of threshold on distance to find the best one.

In [6]:
from components_selection import sweep_connected_components

filtered_embeddings = tf_idf_embeddings[:, np.where(entropy_array < 2.5)[0]]
sweep_connected_components(embedding_matrix = filtered_embeddings)

Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration


In [7]:
pd.read_csv("./results/connected_components.csv")

,Unnamed: 0,Threshold,Connected_components,Largest_component
0,0,0.100000,2979.0,22860.0
1,1,0.144444,3448.0,22334.0
2,2,0.188889,4292.0,21331.0
3,3,0.233333,5542.0,19776.0
4,4,0.277778,7259.0,17251.0
5,5,0.322222,9337.0,13735.0
6,6,0.366667,11600.0,8741.0
7,7,0.411111,13976.0,2517.0
8,8,0.455556,15948.0,132.0
9,9,0.500000,17698.0,52.0


We use a threshold of 0.05: the similarity between papers decreases if we remove the components with higher entropy, so we need to use a lower distance threshold in order to have a giant component big enough in the network.

In [10]:
from components_selection import build_adjacency_matrix

#filter embeddings and build the relative adjacency matrix
filtered_embeddings = tf_idf_embeddings[:, np.where(entropy_array < 2.5)[0]]
print("Number of remaining components: " + str(filtered_embeddings.shape[1]))
adjacency_matrix = build_adjacency_matrix(filtered_embeddings, threshold = 0.05)

#save the adjacency matrix
save_npz("./results/filtered_adjacency_matrix.npz", adjacency_matrix.tocsr())

#build the network corresponding to the adjacency matrix, and count the number of links, the number of connected components
#and the largest connected component
G = nx.from_scipy_sparse_array(adjacency_matrix)
print("Number of links: " + str(G.size()))
print("Number of connected components: " + str(nx.number_connected_components(G)))
print("Size of the largest connected component: " + str(max([len(c) for c in list(nx.connected_components(G))])))

Number of remaining components: 45040
Number of links: 185128
Number of connected components: 2769
Size of the largest connected component: 23084


The network built using tf-idf embeddings filtered by entropy has almost half of the links of the same network built without filtering on entropy: probably a lower number of links means an easier division of the network into communities.

## Division into communities

Before using the algorithms to divide the network into communities, we build the matrix of connections between cathegories, and then for each cathegory we find the group of papers with the highest number of connections with papers of that cathegory.

In [11]:
from components_selection import connectivity_between_cathegories

connectivity_between_cathegories()

Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration


In [12]:
subprocess.run(["Rscript", "./max_connected_cathegory.R"])
pd.read_csv("./results/max_connected_cathegory.csv")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.3     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.0
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
New names:
• `` -> `...1`
Rows: 22 Columns: 24
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): ...1
dbl (23): Accelerator Physics, Applied Physics, Atmospheric and Oceanic Phys...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


,Cathegory,Max_connected_cathegory,Connections
0,Accelerator Physics,Accelerator Physics,668
1,Applied Physics,Optics,3344
2,Atmospheric and Oceanic Physics,Atmospheric and Oceanic Physics,1724
3,Atomic and Molecular Clusters,Chemical Physics,149
4,Atomic Physics,Atomic Physics,1795
5,Biological Physics,Biological Physics,2644
6,Chemical Physics,Chemical Physics,4773
7,Classical Physics,Optics,397
8,Computational Physics,Computational Physics,2312
9,"Data Analysis, Statistics and Probability",Physics and Society,631


Using a filter on entropy does not change so much the connections between cathegories: the majority of cathegories is still connected mostly with papers of the same cathegory.

Now we can use the three algorithms to divide the network into communities.

In [1]:
from components_selection import split_fiedler_eigenvector, split_k_means, split_louvain_method

split_fiedler_eigenvector()
split_k_means()
split_louvain_method()

Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration


We can calculate the number of papers in each one of the communities in which the network has been divided by the three algorithms.

In [ ]:
from components_selection import table_of_papers_by_community, table_of_papers_by_community_louvain

table_of_papers_by_community()
table_of_papers_by_community_louvain()

In [4]:
pd.read_csv("./results/papers_by_community_fiedler_kmeans.csv")

,Unnamed: 0,Fiedler,K_Means
0,0,231,23036
1,1,337,4
2,2,300,2
3,3,300,2
4,4,350,2
5,5,317,3
6,6,317,2
7,7,303,2
8,8,293,3
9,9,424,2


In [6]:
pd.read_csv("./results/papers_by_community_louvain.csv")

,Unnamed: 0,Louvain
0,0,427
1,1,733
2,2,339
3,3,416
4,4,458
5,5,215
6,6,407
7,7,279
8,8,404
9,9,147


There are clearly some important differences with respect to the communities obtained using the non-filtered embeddings. While the K-Means approach still performs very badly, the Fiedler method is able to split the network into communities with a more similar number of nodes in each community. Now we have two communities with the vast majority of nodes instead of 1, and even the other communities have a consistent number of nodes. At the same time, the Louvain algorithm divides the network into 49 communities instead of 19, with a similar number of nodes in each community.

These differences with the case of the network built using non-filtered component is due to the lower number of links: filtering the embedding components with higher entropy reduces the similarity between papers, and so the number of links in the network is lower. This means that the network is more easily separable, and so we obtain more communities, each one with a consistent number of nodes.

If we had kept instead the components with higher entropy, we wuold have obtained the opposite result: the higher entropy components are those corresponding to the words that appear in a bigger number of papers, so removing low entropy components would have made each paper more similar to the others, and thus it would have increased the number of links in the network, making the division into communities much more difficult (and more computationally demanding, too).

Finally, we want to inspect the number of papers of each cathegory in each community. This can be done by looking at the csv files "cathegories_by_community", which are created by the following cell

In [ ]:
from components_selection import cathegories_by_community

cathegories_by_community(split_method = 'fiedler')
cathegories_by_community(split_method = 'kmeans')
cathegories_by_community(split_method = 'louvain')

or directly by using the two following cells to select a community (obtained using respectively Fiedler and Louvain methods) and see the number of papers of each cathegory in that community

In [16]:
#use this cell to see the number of papers of each cathegory in each community (obtained using the Fiedler method)
with open("./results/fiedler_split", "rb") as file:
    subgraphs_list = pickle.load(file)

#use this variable to select the number of the community to consider
community = 5

topics_of_community = nx.get_node_attributes(subgraphs_list[community], "Topic")
number_of_papers_per_topic = dict(Counter(list(topics_of_community.values())))

number_of_papers_per_topic

{'Fluid Dynamics': 29,
 'Atmospheric and Oceanic Physics': 11,
 'Computational Physics': 40,
 'Atomic Physics': 19,
 'Instrumentation and Detectors': 4,
 'Applied Physics': 27,
 'Optics': 87,
 'Physics and Society': 25,
 'History and Philosophy of Physics': 4,
 'Chemical Physics': 18,
 'Classical Physics': 7,
 'Plasma Physics': 7,
 'General Physics': 7,
 'Biological Physics': 11,
 'Data Analysis, Statistics and Probability': 4,
 'Space Physics': 1,
 'Medical Physics': 5,
 'Accelerator Physics': 5,
 'Geophysics': 4,
 'Popular Physics': 2}

In [27]:
#use this cell to see the number of papers of each cathegory in each community (obtained using the Louvain method)
with open("./results/louvain_split", "rb") as file:
    subgraphs_list = pickle.load(file)

#use this variable to select the number of the community to consider
community = 13

topics_of_community = nx.get_node_attributes(subgraphs_list[community], "Topic")
number_of_papers_per_topic = dict(Counter(list(topics_of_community.values())))

number_of_papers_per_topic

{'Optics': 83,
 'Applied Physics': 39,
 'Computational Physics': 73,
 'Classical Physics': 16,
 'General Physics': 24,
 'Plasma Physics': 111,
 'Atomic Physics': 24,
 'Space Physics': 53,
 'Atomic and Molecular Clusters': 4,
 'Chemical Physics': 35,
 'Medical Physics': 10,
 'Fluid Dynamics': 59,
 'Data Analysis, Statistics and Probability': 17,
 'Instrumentation and Detectors': 15,
 'Geophysics': 8,
 'Physics and Society': 10,
 'History and Philosophy of Physics': 9,
 'Biological Physics': 11,
 'Physics Education': 3,
 'Atmospheric and Oceanic Physics': 10,
 'Accelerator Physics': 4,
 'Popular Physics': 1}

Moreover, the two following cells can be used to see the titles of the papers belonging to a certain community

In [18]:
#use this cell to see the titles of the papers of in each community (obtained using the Fiedler method)
with open("./results/fiedler_split", "rb") as file:
    subgraphs_list = pickle.load(file)

#use this variable to select the number of the community to consider
community = 5

titles_of_community = list(nx.get_node_attributes(subgraphs_list[community], "Title").values())

titles_of_community

['Geometry-driven jets underlie dispersal of plants and fungi by raindrops',
 'A Model Intercomparison Study of Mixed-Phase Clouds in a Laboratory Chamber',
 'Energy-Efficient Sampling Using Stochastic Magnetic Tunnel Junctions',
 'Attosecond transient absorption spectroscopy in monolayer hexagonal boron nitride',
 'Sum-of-Gaussians tensor neural networks for high-dimensional Schr\\"odinger equation',
 'Virtual Sensing for Solder Layer Degradation and Temperature Monitoring in IGBT Modules',
 'Permanently magnetized axion-photon conversion surface for direct dark matter searches with BRASS-p',
 'Magnetoresistance effect based on spin-selective transport in nanodevices using chiral molecules',
 'Hidden quantum-classical correspondence in chaotic billiards revealed by mutual information',
 'Coupling opinion dynamics and epidemiology',
 'Wave turbulence, thermalization and multimode locking in optical fibers',
 'Spontaneous symmetry breaking in nonlinear superradiance',
 'Local Wigner-Mas

In [20]:
#use this cell to see the titles of the papers of in each community (obtained using the Louvain method)
with open("./results/louvain_split", "rb") as file:
    subgraphs_list = pickle.load(file)

#use this variable to select the number of the community to consider
community = 5

titles_of_community = list(nx.get_node_attributes(subgraphs_list[community], "Title").values())

titles_of_community

['Dynamics of DNA-Templated Ultrafine Silver Nanowires Formation',
 'A Model Intercomparison Study of Mixed-Phase Clouds in a Laboratory Chamber',
 'Graphene Heterostructure-Based Non-Volatile Memory Devices with Top Floating Gate Programming',
 'On the transpositional relation for nonholonomic systems',
 'On the edge of complexity: The simplest not simple coupled mechanical system',
 'In-vivo femtonewton-sensing nanotribology of Tradescantia zebrina leaf cell inner surface using roll rotation detection',
 'Random Telegraph Noise of MIS and MIOS Silicon Nitride memristors at different resistance states',
 'Combining metal dewetting and lateral etching for the scalable top-down fabrication of GaN nanowire arrays with independently tunable diameter and spacing',
 'Discrete Action, Graph Evolution, and the Hierarchy of Symmetries: A Rigorous Construction of Temporal Layers $C1 \\to C2 \\to C3 \\to C4$',
 'Resonant states and nuclear dynamics in solid-state systems: the case of silicon-hyd

Analyzing the cathegory of papers in each community, we can immediately notice that there is no more one main cathegory for each community: the different cathegories are quite mixed in each community. It looks like the removal of high-entropy components makes it more easy to divide the network, but at the same time the recognition of different topics becomes more difficult. As we can also see from the list of titles in each community, even papers of very different topics are assigned to the same community.

In conclusion, removing high entropy components makes the division into communities easier, but the classification becomes worse. However, we only tried one value of threshold on entropy: maybe removing a lower number of components would give a classification even better than the one obtained keeping all the components.

An important remark is that we don't have any score telling how good is a classification: as already said in the original project, the only way to evaluate the classification performances would be to read each paper (or at least the title/abstract of each paper), and see if papers in the same community actually have a topic in common. Such kind of evaluation, however, requires a lot of time, and so it is very difficult to compare different classifications.